# LeetCode #39 — Combination Sum: Backtracking với chỉ số bắt đầu và dùng lại phần tử

**Mức độ:** Medium · **Chủ đề:** Array, Backtracking

Skeleton trên LeetCode:

```python
class Solution:
    def combinationSum(self, candidates: List[int], target: int) -> List[List[int]]:
```

Ở bài #17 và #22, backtracking sinh **chuỗi** (thứ tự quan trọng). Bài này sinh **tổ hợp** (thứ tự **không** quan trọng: `[2, 2, 3]` và `[3, 2, 2]` là một). Mục tiêu của notebook:

1. Hiểu vì sao backtracking "ngây thơ" sinh ra **tổ hợp trùng** theo thứ tự khác nhau.
2. Kỹ thuật **chỉ số bắt đầu `start`**: chỉ chọn phần tử từ `start` trở đi → mỗi tổ hợp sinh đúng một lần.
3. Cho phép **dùng lại** một phần tử: đệ quy với `i` thay vì `i + 1`.
4. **Cắt tỉa** bằng sắp xếp: số đang xét đã vượt phần còn lại thì dừng cả vòng.

# 1. Bài toán là gì?

Cho mảng `candidates` gồm các số nguyên **khác nhau** và `target`. Tìm **mọi tổ hợp không trùng** của `candidates` có tổng bằng `target`. Mỗi số **được dùng lại không giới hạn** số lần.

Hai tổ hợp khác nhau nếu số lần xuất hiện của ít nhất một số là khác nhau. Thứ tự kết quả tùy ý.

**Constraints (theo đề):**

- `1 <= candidates.length <= 30`
- `2 <= candidates[i] <= 40`, các phần tử khác nhau
- `1 <= target <= 40`
- Đề đảm bảo số tổ hợp nhỏ hơn 150.

---

## Ví dụ

```text
candidates = [2, 3, 6, 7], target = 7  ->  [[2, 2, 3], [7]]
candidates = [2, 3, 5],    target = 8  ->  [[2, 2, 2, 2], [2, 3, 3], [3, 5]]
candidates = [2],          target = 1  ->  []
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Đệ quy (recursion)
2. Quay lui (backtracking)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Đệ quy (recursion)

### Đệ quy là gì?

Một hàm **đệ quy** là hàm tự gọi lại chính nó trên một bài toán **nhỏ hơn** cùng dạng. Mỗi hàm đệ quy có hai phần:

1. **Trường hợp cơ sở (base case)**: bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: chia nhỏ, gọi lại, rồi dùng kết quả để trả lời bài lớn.

Ví dụ `n! = n × (n−1)!`, với `0! = 1`.

In [1]:
def giai_thua(n):
    if n == 0:                  # trường hợp cơ sở
        return 1
    return n * giai_thua(n - 1) # bước đệ quy: bài nhỏ hơn
print([giai_thua(n) for n in range(7)])

[1, 1, 2, 6, 24, 120, 720]


### Ngăn xếp lời gọi (call stack)

Mỗi lần gọi hàm, máy tạo một **khung** (frame) chứa biến cục bộ và đặt lên **ngăn xếp lời gọi**. Hàm con chạy xong thì khung của nó bị bỏ, kết quả trả về cho khung bên dưới. Ta in thụt lề theo độ sâu để thấy rõ:

In [2]:
def giai_thua_vet(n, sau=0):
    le = "    " * sau
    print(f"{le}gọi giai_thua({n})")
    if n == 0:
        print(f"{le}→ trả 1 (cơ sở)")
        return 1
    kq = n * giai_thua_vet(n - 1, sau + 1)
    print(f"{le}→ trả {n} × ... = {kq}")
    return kq
_ = giai_thua_vet(3)

gọi giai_thua(3)
    gọi giai_thua(2)
        gọi giai_thua(1)
            gọi giai_thua(0)
            → trả 1 (cơ sở)
        → trả 1 × ... = 1
    → trả 2 × ... = 2
→ trả 3 × ... = 6


### Cách viết một hàm đệ quy đúng

1. **Định nghĩa rõ** hàm trả về gì cho đầu vào nào ("`f(node)` = chiều cao cây gốc `node`").
2. **Tin** rằng lời gọi trên bài nhỏ hơn đã trả đúng (giả thiết quy nạp).
3. Ghép kết quả các bài nhỏ thành kết quả bài lớn.
4. Kiểm tra trường hợp cơ sở được chạm tới với **mọi** đầu vào (mỗi lần gọi phải nhỏ đi).

Ví dụ: tổng các chữ số, lũy thừa nhanh.

In [3]:
def tong_chu_so(n):
    if n < 10:
        return n
    return n % 10 + tong_chu_so(n // 10)

def luy_thua(x, k):              # x^k, chia đôi số mũ
    if k == 0:
        return 1
    nua = luy_thua(x, k // 2)
    return nua * nua * (x if k % 2 else 1)

print("tong_chu_so(98765) =", tong_chu_so(98765), "| luy_thua(3, 13) =", luy_thua(3, 13), "=", 3 ** 13)

tong_chu_so(98765) = 35 | luy_thua(3, 13) = 1594323 = 1594323


### Ghi nhớ kết quả (memoization)

Một số hàm đệ quy gọi **lặp lại** cùng một bài con rất nhiều lần, ví dụ Fibonacci `F(n) = F(n−1) + F(n−2)`: `F(5)` gọi `F(3)` hai lần, `F(2)` ba lần… → `O(2^n)`.

**Memoization**: lưu kết quả đã tính vào dict (hoặc dùng `@lru_cache`), lần sau gặp lại thì trả ngay → mỗi bài con tính đúng một lần.

In [4]:
from functools import lru_cache

dem_goi = 0
def fib_cham(n):
    global dem_goi
    dem_goi += 1
    return n if n < 2 else fib_cham(n - 1) + fib_cham(n - 2)

memo = {}
def fib_memo(n):
    if n < 2:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

@lru_cache(maxsize=None)
def fib_cache(n):
    return n if n < 2 else fib_cache(n - 1) + fib_cache(n - 2)

print("fib_cham(25) =", fib_cham(25), "với", dem_goi, "lần gọi")
print("fib_memo(25) =", fib_memo(25), "| bảng nhớ có", len(memo), "mục")
print("fib_cache(90) =", fib_cache(90))

fib_cham(25) = 75025 với 242785 lần gọi
fib_memo(25) = 75025 | bảng nhớ có 24 mục
fib_cache(90) = 2880067194370816120


### Lỗi hay gặp

- **Thiếu / sai trường hợp cơ sở** → đệ quy vô hạn → `RecursionError` (Python mặc định giới hạn ~1000 tầng).
- Bài con **không nhỏ đi** (ví dụ gọi `f(n)` trong `f(n)`).
- Đệ quy quá sâu với dữ liệu lớn (danh sách 10^5 nút) → đổi sang vòng lặp hoặc tăng `sys.setrecursionlimit` (có rủi ro).
- Dùng list/biến dùng chung giữa các lời gọi mà quên "trả lại" trạng thái (xem backtracking).

In [5]:
def thieu_co_so(n):
    return thieu_co_so(n - 1)
try:
    thieu_co_so(5)
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "...")

RecursionError: maximum recursion depth exceeded ...


## 2.2 Quay lui (backtracking)

### Quay lui là gì?

Quay lui là cách **liệt kê / tìm kiếm có hệ thống** mọi lời giải bằng cách xây lời giải **từng bước một**. Ở mỗi bước ta thử một lựa chọn, đi tiếp; nếu đi hết (hoặc thấy không thể thành lời giải) thì **hoàn tác** lựa chọn đó và thử lựa chọn khác.

Hình dung bằng **cây quyết định**: mỗi nút là một trạng thái "đã chọn được một phần", mỗi cạnh là một lựa chọn, mỗi lá là một lời giải hoàn chỉnh. Quay lui = DFS trên cây này.

```text
Liệt kê mọi dãy độ dài 2 từ {a, b}:
                 []
            /          \
         [a]            [b]
        /   \          /   \
    [a,a] [a,b]    [b,a] [b,b]      ← 4 lá = 4 lời giải
```

### Khung chuẩn: chọn → đi tiếp → bỏ chọn

```text
def backtrack(trạng_thái):
    if trạng_thái là lời giải hoàn chỉnh:
        ghi lại BẢN SAO
        return
    for lựa_chọn in các lựa chọn hợp lệ:
        chọn          (thêm vào trạng thái)
        backtrack(trạng_thái)
        bỏ chọn       (hoàn tác — trả trạng thái về như cũ)
```

In [6]:
def moi_day(chu, k):
    out, path = [], []
    def bt(sau=0):
        if len(path) == k:
            out.append("".join(path))           # ghi bản sao
            return
        for ch in chu:
            path.append(ch)                     # chọn
            bt(sau + 1)                         # đi tiếp
            path.pop()                          # bỏ chọn
    bt()
    return out

print(moi_day("ab", 2))
print("số dãy độ dài 3 từ 'abc':", len(moi_day("abc", 3)), "= 3^3")

['aa', 'ab', 'ba', 'bb']
số dãy độ dài 3 từ 'abc': 27 = 3^3


### Ba dạng kinh điển

| Dạng | Cách chọn ở mỗi bước | Kích thước kết quả |
|---|---|---|
| **Tập con** (subset) | mỗi phần tử: lấy hoặc không; hoặc chọn phần tử tiếp theo từ chỉ số `start` | `2^n` |
| **Tổ hợp** chập `k` | chọn phần tử từ `start` trở đi (không quay lại phía trước) | `C(n, k)` |
| **Hoán vị** | chọn phần tử **chưa dùng** (mảng `used`) | `n!` |

Tham số `start` đảm bảo không sinh lại cùng một tập theo thứ tự khác (`{1,2}` và `{2,1}` là một).

In [7]:
def tap_con(nums):
    out, path = [], []
    def bt(start):
        out.append(path[:])                     # mọi nút của cây đều là một tập con
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return out

def hoan_vi(nums):
    out, path, used = [], [], [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True; path.append(nums[i])
            bt()
            path.pop(); used[i] = False
    bt()
    return out

print("tập con của [1,2,3]:", tap_con([1, 2, 3]))
print("hoán vị của [1,2,3]:", hoan_vi([1, 2, 3]))

tập con của [1,2,3]: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
hoán vị của [1,2,3]: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### Vết chạy: thấy rõ "chọn / bỏ chọn"

In [8]:
def vet_tap_con(nums):
    path = []
    def bt(start, sau):
        print("    " * sau + f"path = {path}")
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1, sau + 1)
            path.pop()
    bt(0, 0)

vet_tap_con([1, 2, 3])

path = []
    path = [1]
        path = [1, 2]
            path = [1, 2, 3]
        path = [1, 3]
    path = [2]
        path = [2, 3]
    path = [3]


### Cắt tỉa (pruning) và bỏ trùng

- **Cắt tỉa**: nếu trạng thái hiện tại **chắc chắn** không dẫn tới lời giải (tổng đã vượt target, còn quá ít phần tử…), `return` sớm — bỏ cả nhánh cây.
- **Bỏ trùng** khi đầu vào có phần tử lặp: **sắp xếp** trước, rồi trong cùng một tầng (cùng vòng `for`) bỏ qua phần tử bằng phần tử ngay trước nó: `if i > start and nums[i] == nums[i-1]: continue`.

In [9]:
def tap_con_tong(nums, target):
    nums = sorted(nums)
    out, path, dem = [], [], [0]
    def bt(start, con_lai):
        dem[0] += 1
        if con_lai == 0:
            out.append(path[:]); return
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                        # bỏ trùng cùng tầng
            if nums[i] > con_lai:
                break                           # cắt tỉa: đã sắp xếp nên các số sau còn lớn hơn
            path.append(nums[i])
            bt(i + 1, con_lai - nums[i])
            path.pop()
    bt(0, target)
    return out, dem[0]

print("các tập có tổng 8 từ [10,1,2,7,6,1,5]:", tap_con_tong([10, 1, 2, 7, 6, 1, 5], 8))

các tập có tổng 8 từ [10,1,2,7,6,1,5]: ([[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]], 17)


### Lỗi hay gặp

- `out.append(path)` thay vì `out.append(path[:])` → mọi phần tử của `out` là **cùng một** list, cuối cùng đều rỗng.
- Quên **bỏ chọn** (`pop`, `used[i] = False`) → trạng thái "dính" sang nhánh khác.
- Dùng `start = i` thay vì `i + 1` (hoặc ngược lại) → lấy lặp phần tử ngoài ý muốn / thiếu lời giải.
- Bỏ trùng bằng `i > 0` thay vì `i > start` → bỏ mất lời giải hợp lệ dùng hai phần tử bằng nhau ở hai tầng khác nhau.

In [10]:
def sai_khong_sao_chep(nums):
    out, path = [], []
    def bt(start):
        out.append(path)                        # SAI: thêm chính list path
        for i in range(start, len(nums)):
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return out
print("quên sao chép:", sai_khong_sao_chep([1, 2]))

quên sao chép: [[], [], [], []]


## 2.3 Áp dụng vào bài này

Liệt kê tổ hợp có tổng bằng target bằng **quay lui**. Mỗi số được dùng **nhiều lần** → lời gọi đệ quy truyền `start = i` (không phải `i + 1`). **Cắt tỉa** khi tổng còn lại âm (hoặc sắp xếp trước rồi `break` khi số vượt quá).

# 3. Thử backtracking "ngây thơ"

Mỗi bước, chọn **bất kỳ** số nào trong `candidates`, trừ khỏi `remain`, đi tiếp. Dừng khi `remain == 0` (ghi nhận) hoặc `remain < 0` (bỏ).

In [11]:
from typing import List

def naive(candidates: List[int], target: int) -> List[List[int]]:
    result, path = [], []

    def backtrack(remain: int):
        if remain == 0:
            result.append(path[:])
            return
        if remain < 0:
            return
        for x in candidates:
            path.append(x)
            backtrack(remain - x)
            path.pop()

    backtrack(target)
    return result

print(naive([2, 3, 6, 7], 7))

[[2, 2, 3], [2, 3, 2], [3, 2, 2], [7]]


**Vấn đề:** `[2, 2, 3]`, `[2, 3, 2]`, `[3, 2, 2]` là **cùng một tổ hợp**, nhưng được sinh 3 lần vì thứ tự chọn khác nhau.

Có thể sửa bằng cách sắp xếp mỗi kết quả rồi bỏ trùng bằng set — nhưng vẫn lãng phí thời gian đi vào hàng loạt nhánh trùng.

> Tốt hơn: **ép một thứ tự** để mỗi tổ hợp chỉ có **một** cách sinh.

# 4. Ý tưởng: chỉ số bắt đầu `start`

Quy ước: các số trong `path` luôn được chọn theo **thứ tự chỉ số không giảm**. Nghĩa là nếu vừa chọn `candidates[i]`, lần sau chỉ được chọn `candidates[i]`, `candidates[i+1]`, …, **không** quay lại `candidates[0..i-1]`.

```python
def backtrack(start, remain):
    for i in range(start, len(candidates)):
        path.append(candidates[i])
        backtrack(i, remain - candidates[i])     # i, KHÔNG phải i + 1: được dùng lại
        path.pop()
```

- Gọi với `i` (không phải `i + 1`) → số vừa chọn **vẫn được chọn tiếp** (dùng lại không giới hạn).
- Không bao giờ quay lại chỉ số nhỏ hơn → `[3, 2, 2]` không thể sinh ra (vì sau `3` không được chọn `2`).

Mỗi tổ hợp giờ ứng với **đúng một** dãy chỉ số không giảm → sinh đúng một lần.

In [12]:
def combination_sum(candidates: List[int], target: int) -> List[List[int]]:
    result, path = [], []

    def backtrack(start: int, remain: int):
        if remain == 0:
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            x = candidates[i]
            if x > remain:
                continue
            path.append(x)
            backtrack(i, remain - x)
            path.pop()

    backtrack(0, target)
    return result

print(combination_sum([2, 3, 6, 7], 7))
print(combination_sum([2, 3, 5], 8))
print(combination_sum([2], 1))

[[2, 2, 3], [7]]
[[2, 2, 2, 2], [2, 3, 3], [3, 5]]
[]


`result.append(path[:])` lưu **bản sao** của `path` (giống bài #17: lưu tham chiếu sẽ bị `pop` làm rỗng).

# 5. Cắt tỉa mạnh hơn nhờ sắp xếp

Nếu `candidates` được **sắp xếp tăng dần**, khi `candidates[i] > remain` thì mọi số sau `i` cũng `> remain` → dùng `break` thay vì `continue`, bỏ luôn cả phần còn lại của vòng.

In [13]:
def combination_sum_sorted(candidates: List[int], target: int) -> List[List[int]]:
    candidates = sorted(candidates)
    result, path = [], []

    def backtrack(start: int, remain: int):
        if remain == 0:
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            x = candidates[i]
            if x > remain:
                break                       # các số sau còn lớn hơn
            path.append(x)
            backtrack(i, remain - x)
            path.pop()

    backtrack(0, target)
    return result

print(combination_sum_sorted([7, 6, 3, 2], 7))

[[2, 2, 3], [7]]


# 6. Cây quyết định và mô phỏng

`candidates = [2, 3, 6, 7]`, `target = 7`:

```text
                           start=0, remain=7
              /            |              \           \
         +2 (r=5)       +3 (r=4)       +6 (r=1)     +7 (r=0) ✓ [7]
        /    |    \       /    \           (6,7 > 1: dừng)
   +2(r=3) +3(r=2) ... +3(r=1)  ...
    /   \     |
+2(r=1) +3(r=0)✓   +3? (3 > 2: dừng)
 (2>1)  [2,2,3]
```

Nhánh `+3` ở tầng 1 chỉ được chọn tiếp `3, 6, 7` — không quay lại `2`.

In [14]:
def trace_combination_sum(candidates: List[int], target: int) -> List[List[int]]:
    candidates = sorted(candidates)
    result, path = [], []

    def backtrack(start: int, remain: int):
        indent = "  " * len(path)
        if remain == 0:
            print(f"{indent}✓ ghi {path}")
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            x = candidates[i]
            if x > remain:
                print(f"{indent}{x} > {remain}: cắt tỉa (break)")
                break
            path.append(x)
            print(f"{indent}chọn {x} -> path = {path}, còn {remain - x}")
            backtrack(i, remain - x)
            path.pop()

    backtrack(0, target)
    return result

print("Kết quả:", trace_combination_sum([2, 3, 6, 7], 7))

chọn 2 -> path = [2], còn 5


  chọn 2 -> path = [2, 2], còn 3
    chọn 2 -> path = [2, 2, 2], còn 1
      2 > 1: cắt tỉa (break)
    chọn 3 -> path = [2, 2, 3], còn 0
      ✓ ghi [2, 2, 3]
    6 > 3: cắt tỉa (break)
  chọn 3 -> path = [2, 3], còn 2
    3 > 2: cắt tỉa (break)
  6 > 5: cắt tỉa (break)
chọn 3 -> path = [3], còn 4
  chọn 3 -> path = [3, 3], còn 1
    3 > 1: cắt tỉa (break)
  6 > 4: cắt tỉa (break)
chọn 6 -> path = [6], còn 1
  6 > 1: cắt tỉa (break)
chọn 7 -> path = [7], còn 0
  ✓ ghi [7]
Kết quả: [[2, 2, 3], [7]]


# 7. Phiên bản đúng format LeetCode

In [15]:
class Solution:
    def combinationSum(self, candidates: List[int], target: int) -> List[List[int]]:
        candidates.sort()
        result, path = [], []

        def backtrack(start: int, remain: int):
            if remain == 0:
                result.append(path[:])
                return
            for i in range(start, len(candidates)):
                x = candidates[i]
                if x > remain:
                    break
                path.append(x)
                backtrack(i, remain - x)
                path.pop()

        backtrack(0, target)
        return result

In [16]:
def normalize(combos):
    return sorted(sorted(c) for c in combos)

def check(name, actual, expected):
    assert normalize(actual) == normalize(expected), (name, actual, expected)
    assert len(actual) == len({tuple(sorted(c)) for c in actual}), (name, "có trùng")
    print(f"{name:<22} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.combinationSum([2, 3, 6, 7], 7), [[2, 2, 3], [7]])
check("Ví dụ 2", sol.combinationSum([2, 3, 5], 8), [[2, 2, 2, 2], [2, 3, 3], [3, 5]])
check("Ví dụ 3", sol.combinationSum([2], 1), [])
check("Chỉ một cách", sol.combinationSum([5], 10), [[5, 5]])
check("Chưa sắp xếp", sol.combinationSum([8, 7, 4, 3], 11), [[3, 4, 4], [3, 8], [4, 7]])

Ví dụ 1                PASS  [[2, 2, 3], [7]]
Ví dụ 2                PASS  [[2, 2, 2, 2], [2, 3, 3], [3, 5]]
Ví dụ 3                PASS  []
Chỉ một cách           PASS  [[5, 5]]
Chưa sắp xếp           PASS  [[3, 4, 4], [3, 8], [4, 7]]


## Đối chiếu: sinh-rồi-bỏ-trùng vs. chỉ số bắt đầu

In [17]:
import random

def via_naive_dedup(candidates, target):
    return [list(t) for t in {tuple(sorted(c)) for c in naive(candidates, target)}]

random.seed(39)
for _ in range(300):
    cands = random.sample(range(2, 15), random.randint(1, 5))
    t = random.randint(1, 18)
    assert normalize(sol.combinationSum(cands[:], t)) == normalize(via_naive_dedup(cands, t)), (cands, t)
print("300 test ngẫu nhiên: khớp với cách sinh-rồi-bỏ-trùng")

300 test ngẫu nhiên: khớp với cách sinh-rồi-bỏ-trùng


# 8. Vì sao đúng? Độ phức tạp?

**Không bỏ sót:** mọi tổ hợp có tổng `target`, khi viết các phần tử theo thứ tự chỉ số **không giảm**, là một đường đi hợp lệ trong cây (mỗi bước chọn `i >= start`). Cắt tỉa chỉ bỏ những số `> remain` — chúng không thể nằm trong tổ hợp vì mọi số đều dương.

**Không trùng:** mỗi tổ hợp có **duy nhất** một cách viết theo thứ tự chỉ số không giảm → chỉ một đường đi.

**Độ phức tạp:** khó tính chính xác. Một chặn trên thô: gọi `T = target`, `M = min(candidates)`. Độ sâu cây tối đa `T / M`; mỗi nút có tối đa `n` con.

```text
Time:  O(n^(T/M + 1))   (chặn trên rất thô; cắt tỉa làm thực tế nhanh hơn nhiều)
Space: O(T/M) cho path và độ sâu đệ quy
```

Với `T <= 40`, `M >= 2`: độ sâu tối đa 20, và đề đảm bảo < 150 kết quả → chạy rất nhanh.

# 9. Những lỗi dễ mắc

## Lỗi 1: Không dùng `start` → kết quả trùng (mục 3)

## Lỗi 2: Đệ quy với `i + 1` thay vì `i`

Như vậy mỗi số chỉ được dùng **một lần** — đó là bài **#40**, không phải bài này.

In [18]:
def wrong_i_plus_1(candidates: List[int], target: int) -> List[List[int]]:
    result, path = [], []

    def backtrack(start, remain):
        if remain == 0:
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            if candidates[i] > remain:
                continue
            path.append(candidates[i])
            backtrack(i + 1, remain - candidates[i])      # SAI: không cho dùng lại
            path.pop()

    backtrack(0, target)
    return result

print("Sai :", wrong_i_plus_1([2, 3, 6, 7], 7))
print("Đúng:", combination_sum([2, 3, 6, 7], 7))

Sai :

 [[7]]
Đúng: [[2, 2, 3], [7]]


## Lỗi 3: `break` khi chưa sắp xếp

Nếu `candidates = [8, 7, 4, 3]` (chưa sắp xếp) mà dùng `break` khi `x > remain`, ta sẽ dừng ngay ở `8` và bỏ sót `7, 4, 3`.

In [19]:
def wrong_break_unsorted(candidates: List[int], target: int) -> List[List[int]]:
    result, path = [], []

    def backtrack(start, remain):
        if remain == 0:
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            if candidates[i] > remain:
                break                                     # SAI khi chưa sort
            path.append(candidates[i])
            backtrack(i, remain - candidates[i])
            path.pop()

    backtrack(0, target)
    return result

print("Sai :", wrong_break_unsorted([8, 7, 4, 3], 11))
print("Đúng:", combination_sum_sorted([8, 7, 4, 3], 11))

Sai : []
Đúng: [[3, 4, 4], [3, 8], [4, 7]]


## Lỗi 4: `result.append(path)` thay vì bản sao

Mọi phần tử của `result` sẽ trỏ vào cùng một list, cuối cùng đều rỗng.

# 10. Mẹo nhận diện pattern

**Tổ hợp (không quan tâm thứ tự) → backtracking với `start`:**

| Biến thể | Gọi đệ quy | Bỏ trùng |
|---|---|---|
| Dùng lại không giới hạn, phần tử khác nhau (#39) | `backtrack(i, ...)` | không cần |
| Mỗi phần tử một lần, có trùng giá trị (#40) | `backtrack(i + 1, ...)` | sort + bỏ `i > start and a[i] == a[i-1]` |
| Chọn đúng `k` số từ `1..9` (#216) | `backtrack(i + 1, ...)` | không cần |
| Mọi tập con (#78) | `backtrack(i + 1, ...)` | không cần |

**Hoán vị (quan tâm thứ tự) → backtracking với mảng `used`** (#46, #47).

Liên hệ DP: **#518 Coin Change II** đếm **số lượng** tổ hợp — đếm thì dùng DP, liệt kê thì dùng backtracking.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Liệt kê (bằng cây quyết định) mọi tổ hợp cho `candidates = [2, 3, 5]`, `target = 8`. Có bao nhiêu nút bị cắt tỉa bằng `break`?

## Bài 2 — Đoán output

```python
print(len(combination_sum([1, 2], 4)))
print(combination_sum([3, 5], 1))
print(combination_sum([4], 8))
```

(Lưu ý: đề gốc có `candidates[i] >= 2`, nhưng hàm của ta vẫn chạy đúng với `1`.)

## Bài 3 — Điền chỗ trống

In [20]:
def exercise_3_combination_sum(candidates: List[int], target: int) -> List[List[int]]:
    candidates = sorted(candidates)
    result, path = [], []

    def backtrack(start: int, remain: int):
        # TODO 1: base case remain == 0 -> ghi bản sao
        # TODO 2: duyệt i từ start; nếu candidates[i] > remain thì break
        # TODO 3: chọn, đệ quy với i (được dùng lại), bỏ chọn
        pass

    backtrack(0, target)
    return result


# assert normalize(exercise_3_combination_sum([2, 3, 6, 7], 7)) == [[2, 2, 3], [7]]
# assert normalize(exercise_3_combination_sum([2, 3, 5], 8)) == [[2, 2, 2, 2], [2, 3, 3], [3, 5]]

## Bài 4 — Biến thể: Combination Sum III (LeetCode #216)

Tìm mọi tổ hợp gồm **đúng `k`** số khác nhau trong `1..9` có tổng `n`. Mỗi số dùng tối đa một lần.

```python
combination_sum3(3, 7) -> [[1, 2, 4]]
combination_sum3(3, 9) -> [[1, 2, 6], [1, 3, 5], [2, 3, 4]]
combination_sum3(4, 1) -> []
```

In [21]:
def exercise_4_combination_sum3(k: int, n: int) -> List[List[int]]:
    # TODO
    pass


# assert normalize(exercise_4_combination_sum3(3, 7)) == [[1, 2, 4]]
# assert normalize(exercise_4_combination_sum3(3, 9)) == [[1, 2, 6], [1, 3, 5], [2, 3, 4]]
# assert exercise_4_combination_sum3(4, 1) == []

## Bài 5 — Mini challenge: đếm bằng DP (LeetCode #518 Coin Change II)

Đếm **số** tổ hợp (không liệt kê) của `coins` có tổng `amount`, mỗi đồng dùng không giới hạn.

```python
count_combinations(5, [1, 2, 5]) -> 4
count_combinations(3, [2])       -> 0
count_combinations(10, [10])     -> 1
```

Gợi ý nhanh: `dp[s]` = số cách tạo tổng `s`. Duyệt **từng đồng xu ở vòng ngoài**, tổng ở vòng trong — thứ tự vòng lặp này chính là ý tưởng "chỉ số bắt đầu" của backtracking!

In [22]:
def exercise_5_count_combinations(amount: int, coins: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_count_combinations(5, [1, 2, 5]) == 4
# assert exercise_5_count_combinations(3, [2]) == 0
# assert exercise_5_count_combinations(10, [10]) == 1

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_combination_sum([2, 3, 5], 8)` sau khi tự vẽ để so sánh.

**Bài 3**
- Mức 1: so với `combination_sum_sorted` ở mục 5.

**Bài 4**
- Mức 1: `candidates = 1..9`, gọi đệ quy với `i + 1`.
- Mức 2: base case: `len(path) == k`; ghi nhận chỉ khi `remain == 0`.

**Bài 5**
- Mức 1: `dp = [1] + [0] * amount` (một cách tạo tổng 0: không chọn gì).
- Mức 2: `for coin in coins: for s in range(coin, amount + 1): dp[s] += dp[s - coin]`.
- Mức 3: nếu đảo hai vòng lặp, bạn sẽ đếm **hoán vị** (thứ tự khác nhau tính khác) — đó là bài #377.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [23]:
print("Kết quả:", trace_combination_sum([2, 3, 5], 8))

chọn 2 -> path = [2], còn 6
  chọn 2 -> path = [2, 2], còn 4
    chọn 2 -> path = [2, 2, 2], còn 2
      chọn 2 -> path = [2, 2, 2, 2], còn 0
        ✓ ghi [2, 2, 2, 2]
      3 > 2: cắt tỉa (break)
    chọn 3 -> path = [2, 2, 3], còn 1
      3 > 1: cắt tỉa (break)
    5 > 4: cắt tỉa (break)
  chọn 3 -> path = [2, 3], còn 3
    chọn 3 -> path = [2, 3, 3], còn 0
      ✓ ghi [2, 3, 3]
    5 > 3: cắt tỉa (break)
  chọn 5 -> path = [2, 5], còn 1
    5 > 1: cắt tỉa (break)
chọn 3 -> path = [3], còn 5
  chọn 3 -> path = [3, 3], còn 2
    3 > 2: cắt tỉa (break)
  chọn 5 -> path = [3, 5], còn 0
    ✓ ghi [3, 5]
chọn 5 -> path = [5], còn 3
  5 > 3: cắt tỉa (break)
Kết quả: [[2, 2, 2, 2], [2, 3, 3], [3, 5]]


## Đáp án Bài 2

In [24]:
print(len(combination_sum([1, 2], 4)))   # 3: [1,1,1,1], [1,1,2], [2,2]
print(combination_sum([3, 5], 1))        # []
print(combination_sum([4], 8))           # [[4, 4]]

3
[]
[[4, 4]]


## Đáp án Bài 3

In [25]:
def answer_3_combination_sum(candidates: List[int], target: int) -> List[List[int]]:
    candidates = sorted(candidates)
    result, path = [], []

    def backtrack(start: int, remain: int):
        if remain == 0:
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            if candidates[i] > remain:
                break
            path.append(candidates[i])
            backtrack(i, remain - candidates[i])
            path.pop()

    backtrack(0, target)
    return result

assert normalize(answer_3_combination_sum([2, 3, 6, 7], 7)) == [[2, 2, 3], [7]]
assert normalize(answer_3_combination_sum([2, 3, 5], 8)) == [[2, 2, 2, 2], [2, 3, 3], [3, 5]]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [26]:
def answer_4_combination_sum3(k: int, n: int) -> List[List[int]]:
    result, path = [], []

    def backtrack(start: int, remain: int):
        if len(path) == k:
            if remain == 0:
                result.append(path[:])
            return
        for x in range(start, 10):
            if x > remain:
                break
            path.append(x)
            backtrack(x + 1, remain - x)
            path.pop()

    backtrack(1, n)
    return result

assert normalize(answer_4_combination_sum3(3, 7)) == [[1, 2, 4]]
assert normalize(answer_4_combination_sum3(3, 9)) == [[1, 2, 6], [1, 3, 5], [2, 3, 4]]
assert answer_4_combination_sum3(4, 1) == []
print("Bài 4 PASS")

Bài 4 PASS

## Đáp án Bài 5

In [27]:
def answer_5_count_combinations(amount: int, coins: List[int]) -> int:
    dp = [1] + [0] * amount
    for coin in coins:
        for s in range(coin, amount + 1):
            dp[s] += dp[s - coin]
    return dp[amount]

assert answer_5_count_combinations(5, [1, 2, 5]) == 4
assert answer_5_count_combinations(3, [2]) == 0
assert answer_5_count_combinations(10, [10]) == 1
for _ in range(200):
    cands = random.sample(range(2, 12), random.randint(1, 4))
    t = random.randint(1, 20)
    assert answer_5_count_combinations(t, cands) == len(sol.combinationSum(cands[:], t))
print("Bài 5 PASS (số đếm bằng DP khớp số tổ hợp backtracking liệt kê)")

Bài 5 PASS (số đếm bằng DP khớp số tổ hợp backtracking liệt kê)


# 14. Tổng kết

> Tổ hợp → **ép thứ tự** bằng chỉ số `start`. Được dùng lại → đệ quy với `i`. Sắp xếp trước để **cắt tỉa bằng `break`** khi số đang xét vượt phần còn lại.

---

## Checklist trước khi nộp

- [ ] Vòng lặp `for i in range(start, n)`?
- [ ] Đệ quy với `i` (dùng lại), không phải `i + 1`?
- [ ] Đã sắp xếp nếu dùng `break`?
- [ ] Lưu `path[:]`, không lưu `path`?
- [ ] Đã thử trường hợp không có lời giải (`[2]`, `1`)?